# MXP_scheduler 탐색 노트북

`mxp_scheduler.py` 의 입력 클래스/헬퍼를 **직접 숫자 넣어보며** 확인하는 노트북.

다루는 범위 (코드 읽은 순서대로): `HW` -> `Work` -> `Mapping` -> `_out_in`.

**사용법**: 각 섹션의 `### 바꿔보기` 셀에서 변수만 고치고 Shift+Enter 로 다시 실행하면 됨.

> 참고: 클래스 자체는 디스크에 아무것도 "저장"하지 않는다. 메모리상 객체일 뿐이고,
> 결과를 파일/화면으로 내보내는 건 맨 끝 CLI 의 `report()` (stdout 출력) 가 전부다.
> 이 노트북에서는 `print()` 로 객체 상태를 들여다본다.

In [1]:
# 셋업: 이 노트북이 MXP_scheduler/ 안에 있으므로 그대로 import 됨
import os, sys
sys.path.insert(0, os.path.abspath("."))
import importlib
import mxp_scheduler as s
importlib.reload(s)   # 코드 수정 후 재실행하면 반영됨
print("loaded:", s.__file__)
print("TILE =", s.TILE, "| FP32_BITS =", s.FP32_BITS)
print("DEFAULT_COEFFS =", s.DEFAULT_COEFFS)

loaded: c:\Users\ptj72\Desktop\Desktop\00project\gemm_sram\MXP_scheduler\mxp_scheduler.py
TILE = 32 | FP32_BITS = 32
DEFAULT_COEFFS = {'dram': 200.0, 'onchip': 6.0, 'mac': 1.0, 'rmw': 5.0}


## 1. `HW` - 칩 한 대의 스펙

- `bank_size x banks x word_bits` -> `cap_bits` (SRAM 총 비트, feasibility 게이트 기준)
- `dram_bw / freq_ratio` -> `eff_bw` (DRAM 대역폭을 on-chip cycle 단위로 환산)
- `coeffs` -> 옵티마이저 목적함수 가중치 (dram/onchip/mac/rmw)

음수/0/오타키는 `__post_init__` 이 즉시 거부한다 (아래 섹션 5에서 실험).

In [2]:
### 바꿔보기: HW 파라미터
bank_size  = 1024
banks      = 32
dram_bw    = 64      # bits per DRAM cycle
freq_ratio = 1.0     # f_chip / f_dram  (2.0 면 칩이 DRAM 보다 2배 빠름)
word_bits  = 32

hw = s.HW(bank_size=bank_size, banks=banks, dram_bw=dram_bw,
          freq_ratio=freq_ratio, word_bits=word_bits)

print("repr      :", hw)
print("cap_bits  :", hw.cap_bits, f"  (= {bank_size} * {banks} * {word_bits})")
print("eff_bw    :", hw.eff_bw, f"  (= dram_bw / freq_ratio = {dram_bw} / {freq_ratio})")
print("coeffs    :", hw.coeffs)
print()
print("# freq_ratio 를 2.0 으로 바꾸면 eff_bw 가 절반이 된다 (전송이 on-chip cycle 로 2배 길어짐)")

repr      : HW(bank_size=1024, banks=32, dram_bw=64, word_bits=32, freq_ratio=1.0, coeffs={'dram': 200.0, 'onchip': 6.0, 'mac': 1.0, 'rmw': 5.0})
cap_bits  : 1048576   (= 1024 * 32 * 32)
eff_bw    : 64.0   (= dram_bw / freq_ratio = 64 / 1.0)
coeffs    : {'dram': 200.0, 'onchip': 6.0, 'mac': 1.0, 'rmw': 5.0}

# freq_ratio 를 2.0 으로 바꾸면 eff_bw 가 절반이 된다 (전송이 on-chip cycle 로 2배 길어짐)


## 2. `Work` - 풀어야 할 GEMM 한 개

`C = W . A`,  `M/K/N` 은 형상 (각 32 의 배수).

- `wbits` : `MT x KT` 행렬 = 타일별 **평균 weight 비트** (mixed-precision 의 표현 수단). 분수 허용, `[2,8]`.
- `act_bits` : activation 정밀도 (layer 균일, 2/4/8).

`MT=M/32`, `KT=K/32`, `NT=N/32`. `total_w_bits = 32*32 * sum(모든 wbits)`.

In [ ]:
### 바꿔보기: Work (GEMM) 파라미터
M, K, N  = 128, 128, 128
act_bits = 8

MT, KT, NT = M // s.TILE, K // s.TILE, N // s.TILE

# --- wbits 만드는 법 3가지 (하나만 골라서 쓰면 됨) ---
# (a) 균일: 모든 타일 동일 비트
#wbits = [[act_bits] * KT for _ in range(MT)]

# (b) 직접 mixed: MT x KT 를 손으로 (M=K=64 예시면 2x2)
wbits = [[2, 8, 4, 8],
         [8, 2, 8, 4],
         [4, 8, 2, 8],
         [8, 4, 8, 2]]

# (c) 분수 평균도 가능 (타일은 32 블록의 평균이라 자연스러움)
# wbits = [[3.5, 8.0], [6.0, 2.0]]

w = s.Work(M=M, K=K, N=N, wbits=wbits, act_bits=act_bits)

print("MT, KT, NT  :", w.MT, w.KT, w.NT)
print("wbits       :", w.wbits)
print("total_w_bits:", w.total_w_bits,
      f"  (= 32*32 * sum(wbits) = 1024 * {sum(sum(r) for r in w.wbits)})")

MT, KT, NT  : 2 2 2
wbits       : [[2, 8], [8, 2]]
total_w_bits: 20480   (= 32*32 * sum(wbits) = 1024 * 20)


## 3. `Mapping` - 스케줄 후보 하나

- `perm` : 루프 순서. `perm[0]` 이 최외곽. M/K/N 의 순열 (최대 6 가지).
- `m_in / k_in / n_in` : 각 차원에서 SRAM 에 **동시 상주**시킬 inner 타일 수 (blocking).

`frozen=True` 라 불변. `perm` 이 진짜 순열이 아니면 생성 시 거부.

In [ ]:
### 바꿔보기: Mapping (스케줄 후보)
perm = ("M", "K", "N")   # M 최외곽, N 최내곽
m_in = 1                  # M 상주 타일 수 (w.MT 의 약수여야 함)
k_in = 2                  # K 상주 타일 수 (w.KT 의 약수)
n_in = 4                  # N 상주 타일 수 (w.NT 의 약수)

m = s.Mapping(perm=perm, m_in=m_in, k_in=k_in, n_in=n_in)
print("repr :", m)
print("perm :", m.perm, f" ({m.perm[0]} outermost, {m.perm[-1]} innermost)")
print()
print("# 참고: 유효한 inner 값(=약수)이 뭔지 보려면:")
print("  MT 약수:", s.divisors(w.MT))
print("  KT 약수:", s.divisors(w.KT))
print("  NT 약수:", s.divisors(w.NT))

## 4. `_out_in(m, w)` - inner(상주) -> outer(반복) 역산

불변식: **`out[dim] * inn[dim] = 전체 타일 수`**.

inner 를 크게 잡으면 outer 반복이 줄고(상주 많음), 작게 잡으면 outer 가 늘어남(DRAM 재로드 많음).
`out[dim] == 1` 은 "그 차원 통째로 상주, 바깥 루프 없음" -> 나중에 C spill 판정의 핵심.

In [ ]:
### 바꿔보기: _out_in 결과 보기 (위 m, w 를 그대로 사용)
out, inn = s._out_in(m, w)
print("inn (상주) :", inn)
print("out (반복) :", out, "  (= 전체타일 // inn)")
print("check      :", {d: out[d] * inn[d] for d in ("M", "K", "N")},
      f" == (MT,KT,NT)=({w.MT},{w.KT},{w.NT}) ?")
print()
# inner 별 out 이 어떻게 변하는지 한눈에 (K 차원 예시)
print(f"K 차원 (KT={w.KT}) 의 inner->outer 표:")
for ki in s.divisors(w.KT):
    print(f"  k_in={ki:>2}  ->  out['K']={w.KT // ki:>2}   (상주 {ki}타일, 바깥 {w.KT // ki}번 반복)")

## 5. 검증/에러 실험실

각 클래스의 `__post_init__` / `_out_in` 가드가 잘못된 입력을 어떻게 막는지 직접 깨보기.
아래 `try_it` 로 ValueError 메시지를 확인할 수 있다.

특히 `k_in=3 (KT=4)` 같은 **약수 아닌 blocking** 은 거부된다 -- ragged blocking 표현 불가가
이 모델의 알려진 한계 (소수 타일수 KT=5,7 에서 진짜 옵션 누락). 추후 수정 후보.

In [ ]:
def try_it(label, fn):
    try:
        r = fn()
        print(f"[OK ] {label}: {r}")
    except ValueError as e:
        print(f"[REJ] {label}: {e}")

print("=== divisor guard (모델 한계) ===")
try_it("k_in=3 (KT=4, ragged)", lambda: s._out_in(s.Mapping(("M","K","N"), 1, 3, 1), w))
try_it("k_in=2 (KT=4, divisor)", lambda: s._out_in(s.Mapping(("M","K","N"), 1, 2, 1), w))

print("\n=== Work 검증 ===")
try_it("M=100 (32 배수 아님)", lambda: s.Work(100, 128, 128, [[8]*4 for _ in range(4)], 8))
try_it("wbits=9 ([2,8] 벗어남)", lambda: s.Work(64, 64, 64, [[2,9],[8,2]], 8))
try_it("wbits=3.7 (분수 OK)", lambda: s.Work(64, 64, 64, [[3.7,8],[8,2]], 8).wbits)

print("\n=== HW 검증 ===")
try_it("coeffs 오타 'darm'", lambda: s.HW(1024,32,64, coeffs={"darm":200,"onchip":6,"mac":1,"rmw":5}))
try_it("dram_bw=0", lambda: s.HW(1024,32,0))
try_it("freq_ratio=-1", lambda: s.HW(1024,32,64, freq_ratio=-1))

## 6. 여기까지의 객체로 다음 단계 미리 맛보기 (선택)

아직 코드로 안 읽은 함수들이지만, 위에서 만든 `m, w, hw` 를 그대로 넣으면 동작한다.
다음 세션에 `_blocks` -> `dram_bits` -> `energy_breakdown` -> stall 순으로 읽을 예정.

In [ ]:
print("feasible      :", s.feasible(m, w, hw))
print("footprint_bits:", s.footprint_bits(m, w), "<= cap_bits", hw.cap_bits)
print("dram_bits     :", s.dram_bits(m, w))
print("compute_work  :", s.compute_work(w))
print()
ev = s.evaluate(m, w, hw)
print("evaluate() 한 줄 요약:")
for k in ("feasible", "energy", "compute_work", "stall", "fill", "actual_cycle"):
    print(f"  {k:>13}: {ev[k]}")

In [ ]:
# 전체 최적화도 한 줄: 위 w, hw 로 모든 후보 랭킹 -> 상위 10개 표
ranked = s.optimize(w, hw)
print(s.report(ranked, w, hw))